# Qwen2.5-3B on SageMaker

This notebook runs on an AWS SageMaker Notebook Instance.
It shows how we deploy and call the Qwen2.5-3B-Instruct model using a real-time (synchronous) endpoint. We use it to generate scenarios directly from the backend during gameplay.

> ⚠️ Note: You’ll need AWS credentials with SageMaker permissions to run this (see documentation deliverable)
> The copy in GitHub is for reference and won’t execute without our configured AWS environment.

What it does

- creates a realtime TGI endpoint (ml.g5.xlarge)

- calls the endpoint with invoke_endpoint and returns the text response

- includes simple teardown helpers to delete the endpoint, config, and model


## 1) Config
- Default model: Qwen2.5-3B Instruct
- Instance: ml.g5.xlarge

In [25]:
REGION = "us-west-2"

import boto3, botocore, os, json, time, typing
from sagemaker import get_execution_role
from sagemaker.huggingface import HuggingFaceModel
from sagemaker.session import Session

boto_sess = boto3.Session(region_name=REGION)
sm  = boto_sess.client("sagemaker")
rt  = boto_sess.client("sagemaker-runtime")
sagemaker_sess = Session(boto_session=boto_sess)

ACCOUNT = boto3.client("sts").get_caller_identity()["Account"]
ROLE = get_execution_role()

# Model / endpoint names
MODEL_ID      = "Qwen/Qwen2.5-3B-Instruct"
ENDPOINT_NAME = "neuro-rag-rt"
INSTANCE_TYPE = "ml.g5.xlarge"

# TGI token safety
MAX_INPUT_TOKENS = 1536
MAX_TOTAL_TOKENS = 2048

IMAGE_URI = f"763104351884.dkr.ecr.{REGION}.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04"

print("Region:", REGION)
print("Model:", MODEL_ID)
print("Image:", IMAGE_URI)
print("Endpoint:", ENDPOINT_NAME)

env = {
    "HF_MODEL_ID": MODEL_ID,
    "HF_TASK": "text-generation",
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "MAX_INPUT_TOKENS": str(MAX_INPUT_TOKENS),
    "MAX_TOTAL_TOKENS": str(MAX_TOTAL_TOKENS),
}

Region: us-west-2
Model: Qwen/Qwen2.5-3B-Instruct
Image: 763104351884.dkr.ecr.us-west-2.amazonaws.com/huggingface-pytorch-tgi-inference:2.4.0-tgi2.4.0-gpu-py311-cu124-ubuntu22.04
Endpoint: neuro-rag-rt


## 2) Helpers
- `safe_call`: safely wraps AWS SDK calls and skips missing-resource or validation errors 
- `kill`: idempotent teardown (endpoint → config → model)
- `wait_deleted`: waits until endpoint fully deleted

In [26]:
def safe_call(fn, **kw):
    try:
        return fn(**kw)
    except botocore.exceptions.ClientError as e:
        code = e.response.get("Error", {}).get("Code")
        if code in {"ValidationException", "ResourceNotFound"}:
            return None
        raise

def kill(name: str):
    print(f"Deleting endpoint (if exists): {name}")
    safe_call(sm.delete_endpoint, EndpointName=name)
    print(f"Deleting endpoint config (if exists): {name}")
    safe_call(sm.delete_endpoint_config, EndpointConfigName=name)
    print(f"Deleting model (if exists): {name}")
    safe_call(sm.delete_model, ModelName=name)

def wait_deleted(name: str, timeout_min=15):
    t0 = time.time()
    while True:
        try:
            sm.describe_endpoint(EndpointName=name)
        except botocore.exceptions.ClientError as e:
            if e.response.get("Error", {}).get("Code") == "ValidationException":
                print("Endpoint deleted:", name)
                return True
            raise
        if time.time() - t0 > timeout_min*60:
            print("Timed out waiting for deletion.")
            return False
        time.sleep(8)

## 3) Clean slate & DEPLOY
- Start from a clean slate, delete any old endpoint with the same name 
- Deploy new Qwen2.5-3B model as a realtime SageMaker endpoint

In [27]:
kill(ENDPOINT_NAME)

hf_model = HuggingFaceModel(
    image_uri=IMAGE_URI,
    role=ROLE,
    env=env,
    sagemaker_session=sagemaker_sess,
)

predictor = hf_model.deploy(
    endpoint_name=ENDPOINT_NAME,
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    container_startup_health_check_timeout=1800,
    wait=True,  # block until InService
)

desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
print("EndpointStatus:", desc["EndpointStatus"])


Deleting endpoint (if exists): neuro-rag-rt
Deleting endpoint config (if exists): neuro-rag-rt
Deleting model (if exists): neuro-rag-rt
-----------!EndpointStatus: InService


## 4) Simple synchronous client
- Helper that prompts directly to the realtime endpoint, returns the generated text

In [28]:
import time

def invoke_sync_tgi(
    prompt: str,
    *,
    max_new_tokens: int = 180,
    temperature: float = 0.7,
    return_full_text: bool = False,
    top_p: float = 0.9,
    top_k: int | None = None,
    repetition_penalty: float | None = None,
    retries: int = 3,
    backoff_s: float = 1.0,
) -> str:
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": max_new_tokens,
            "temperature": temperature,
            "return_full_text": return_full_text,
            "top_p": top_p,
        },
    }
    if top_k is not None:
        payload["parameters"]["top_k"] = top_k
    if repetition_penalty is not None:
        payload["parameters"]["repetition_penalty"] = repetition_penalty

    last_err = None
    for attempt in range(retries):
        try:
            resp = rt.invoke_endpoint(
                EndpointName=ENDPOINT_NAME,
                ContentType="application/json",
                Body=json.dumps(payload).encode("utf-8"),
            )
            body = resp["Body"].read()
            try:
                data = json.loads(body)
            except Exception:
                return body.decode("utf-8", errors="replace")

            if isinstance(data, list) and data and isinstance(data[0], dict) and "generated_text" in data[0]:
                return data[0]["generated_text"]
            if isinstance(data, dict) and "generated_text" in data:
                return data["generated_text"]
            return json.dumps(data)
        except botocore.exceptions.ClientError as e:
            last_err = e
            time.sleep(backoff_s)
            backoff_s *= 1.5
    raise last_err or RuntimeError("invoke failed")

## 5) Quick test
- Simple end-to-end to confirm everything is wired up.

In [29]:
test_prompt = (
    "Write a ~40 word opening scenario set in 2075 about memory implants and ethics. "
    "Output JSON with keys: scenario_text, choices (3 items), citations (empty array is fine)."
)
out = invoke_sync_tgi(test_prompt, max_new_tokens=180, temperature=0.7)
print(out[:800])

 JSON:
{
    "scenario_text": "In 2075, society has embraced memory implants, allowing people to enhance their lives with perfect memories or erase traumatic events. But as the technology advances, ethical debates rage about who should have access to these powerful tools and what limits should be set.",
    "choices": [
        "Prohibit all memory implants to prevent the manipulation of memories and protect individual freedom.",
        "Allow unrestricted access to memory implants but mandate regular mental health checks to ensure no harm is done.",
        "Keep memory implants available but require detailed ethical guidelines and public oversight to guide their use."
    ],
    "citations": []
} The provided JSON structure has been followed, crafting a scenario that discusses the ethic


## 6) Teardown
- This deletes the endpoint, config, and model.


In [7]:
print("Tearing down:", ENDPOINT_NAME)
kill(ENDPOINT_NAME)
wait_deleted(ENDPOINT_NAME)

safe_call(sm.delete_endpoint_config, EndpointConfigName=ENDPOINT_NAME)
safe_call(sm.delete_model, ModelName=ENDPOINT_NAME)
print("Cleanup done.")

Tearing down: neuro-rag-rt
Deleting endpoint (if exists): neuro-rag-rt
Deleting endpoint config (if exists): neuro-rag-rt
Deleting model (if exists): neuro-rag-rt
Endpoint deleted: neuro-rag-rt
Cleanup done.
